# CP2 Week 3 -- Configurable Cleaning Pipeline

**Course:** Computer Programming 2 (CP2)
**Prerequisites:** Weeks 1-2 (schema validation, dict analytics)
**Focus:** rule registry, config-driven cleaning steps

## Learning Objectives
- Design a config-driven cleaning pipeline
- Create a rule registry pattern
- Write individual cleaning rules as functions
- Track drop reasons for every removed row
- Add custom rules without changing core code

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Why Config-Driven Cleaning?

In CP1, your cleaning code probably looked like this:

```python
# Hard-coded cleaning (v1 style)
if row['value'] == '':
    continue  # skip missing
if float(row['value']) < 0:
    continue  # skip negative
```

This works, but has problems:
- Every track has DIFFERENT rules (robotics cares about temperature range, space cares about brightness)
- Changing a threshold means editing code deep inside a function
- You cannot easily turn rules on/off for testing

**Config-driven cleaning** separates WHAT to clean (config) from HOW to clean (code).

### Config-Driven Architecture

```
CONFIG (data, easy to change)      CODE (logic, rarely changes)
+--------------------------+       +---------------------------+
| drop_missing: True       | ----> | if config['drop_missing']:|
| numeric_columns: [value] |       |   check_missing(row, col) |
| value_ranges:            |       |                           |
|   value: (0, 100)        | ----> | if val < low or val > high|
| strip_whitespace: True   |       |   drop(row, reason)       |
+--------------------------+       +---------------------------+
```

### Step 1: Create a Cleaning Config

In [ ]:
def create_cleaning_config():
    """Create a configuration for the cleaning pipeline.
    
    All cleaning behavior is controlled by this dict.
    Change the config to change the behavior -- no code changes needed.
    """
    return {
        # Which checks to run
        "drop_missing": True,
        "drop_duplicates": True,
        "check_types": True,
        "check_ranges": True,
        
        # Column definitions
        "numeric_columns": ["value", "score"],
        "string_columns": ["status", "category"],
        
        # Acceptable ranges
        "value_ranges": {
            "value": (0, 100),
            "score": (0, 100),
        },
        
        # Text processing
        "strip_whitespace": True,
        "lowercase_strings": False,
        
        # Custom rules to apply
        "custom_rules": ["remove_negative", "cap_outliers"],
    }

config = create_cleaning_config()
print("Cleaning config:")
for key, val in config.items():
    print("  " + str(key) + ": " + str(val))

**Expected Output:**
```
Cleaning config:
  drop_missing: True
  drop_duplicates: True
  check_types: True
  check_ranges: True
  numeric_columns: ['value', 'score']
  string_columns: ['status', 'category']
  value_ranges: {'value': (0, 100), 'score': (0, 100)}
  strip_whitespace: True
  lowercase_strings: False
  custom_rules: ['remove_negative', 'cap_outliers']
```

### Why This Matters

Every professional data pipeline uses configuration. It lets you:

1. Change behavior without changing code
2. Run the same pipeline on different datasets with different rules
3. Share configs with teammates ("use this config for the March data")
4. Version control your configs separately from your code

---
## Part 2: Rule-Based Cleaning Pipeline

Now let us build the actual cleaning function that reads the config:

In [ ]:
def apply_cleaning_rules(data, config):
    """Apply configurable cleaning rules to data.
    
    Args:
        data: list of dicts (raw data)
        config: cleaning configuration dict
    
    Returns:
        tuple: (cleaned_data, cleaning_report)
    """
    cleaned = []
    report = {
        "total": len(data),
        "kept": 0,
        "dropped": {},
    }
    
    for row in data:
        skip_reason = None
        
        # Rule 1: Strip whitespace
        if config.get("strip_whitespace", False):
            for key in row:
                if isinstance(row[key], str):
                    row[key] = row[key].strip()
        
        # Rule 2: Missing values
        if config.get("drop_missing", True) and not skip_reason:
            for col in config.get("numeric_columns", []):
                val = row.get(col, "")
                if val is None or str(val).strip() == "":
                    skip_reason = "missing_" + col
                    break
        
        # Rule 3: Type conversion + range check
        if config.get("check_types", True) and not skip_reason:
            for col in config.get("numeric_columns", []):
                if col in row:
                    try:
                        val = float(row[col])
                        # Range check
                        if config.get("check_ranges", True):
                            ranges = config.get("value_ranges", {})
                            if col in ranges:
                                low, high = ranges[col]
                                if val < low or val > high:
                                    skip_reason = "out_of_range_" + col
                                    break
                        row[col] = val  # store as float
                    except (ValueError, TypeError):
                        skip_reason = "non_numeric_" + col
                        break
        
        # Record result
        if skip_reason:
            report["dropped"][skip_reason] = report["dropped"].get(skip_reason, 0) + 1
        else:
            cleaned.append(row)
    
    report["kept"] = len(cleaned)
    
    # Print summary
    print("Cleaning: " + str(report["total"]) + " -> " + str(report["kept"]) + " rows")
    for reason, count in sorted(report["dropped"].items()):
        print("  Dropped (" + reason + "): " + str(count))
    
    return cleaned, report

# Test
data = [
    {"id": 1, "value": "25",  "score": "80"},
    {"id": 2, "value": "",    "score": "90"},      # missing value
    {"id": 3, "value": "abc", "score": "75"},      # non-numeric
    {"id": 4, "value": "50",  "score": "150"},     # score out of range
    {"id": 5, "value": "30",  "score": "85"},
    {"id": 6, "value": " 45 ","score": " 70 "},   # whitespace
]

config = create_cleaning_config()
cleaned, report = apply_cleaning_rules(data, config)
print("\nKept rows:")
for row in cleaned:
    print("  " + str(row))

**Expected Output:**
```
Cleaning: 6 -> 3 rows
  Dropped (missing_value): 1
  Dropped (non_numeric_value): 1
  Dropped (out_of_range_score): 1

Kept rows:
  {'id': 1, 'value': 25.0, 'score': 80.0}
  {'id': 5, 'value': 30.0, 'score': 85.0}
  {'id': 6, 'value': 45.0, 'score': 70.0}
```

---
## Part 3: The Rule Registry Pattern

The **rule registry** lets you add custom cleaning rules as functions without modifying the core cleaning code. Each rule is a function with a standard signature.

In [ ]:
# Custom cleaning rules -- each takes (row, config) and returns
# a drop reason string or None (if the row should be kept)

def rule_remove_negative(row, config):
    """Remove rows with any negative numeric value."""
    for col in config.get("numeric_columns", []):
        if col in row and isinstance(row[col], (int, float)):
            if row[col] < 0:
                return "negative_" + col
    return None

def rule_cap_outliers(row, config):
    """Cap extreme values instead of dropping.
    
    This is a "soft" rule -- it modifies instead of dropping.
    Returns None (never drops), but clamps values to range.
    """
    for col in config.get("numeric_columns", []):
        if col in row and isinstance(row[col], (int, float)):
            ranges = config.get("value_ranges", {})
            if col in ranges:
                low, high = ranges[col]
                row[col] = max(low, min(high, row[col]))
    return None  # never drops

def rule_remove_duplicates(row, config, seen=None):
    """Remove duplicate rows based on ID column."""
    if seen is None:
        seen = set()
    id_val = row.get("id")
    if id_val in seen:
        return "duplicate_id"
    seen.add(id_val)
    return None

# The Rule Registry: maps rule names to functions
RULE_REGISTRY = {
    "remove_negative": rule_remove_negative,
    "cap_outliers": rule_cap_outliers,
    "remove_duplicates": rule_remove_duplicates,
}

print("Available rules:")
for name in RULE_REGISTRY:
    print("  " + name + ": " + RULE_REGISTRY[name].__doc__.strip().split(chr(10))[0])

**Expected Output:**
```
Available rules:
  remove_negative: Remove rows with any negative numeric value.
  cap_outliers: Cap extreme values instead of dropping.
  remove_duplicates: Remove duplicate rows based on ID column.
```

### Using the Rule Registry

In [ ]:
def apply_custom_rules(data, config, registry=None):
    """Apply custom rules from the config using the registry.
    
    Args:
        data: list of dicts (already passed basic cleaning)
        config: must have "custom_rules" list of rule names
        registry: dict mapping rule names to functions
    
    Returns:
        tuple: (cleaned_data, rule_report)
    """
    if registry is None:
        registry = RULE_REGISTRY
    
    rule_names = config.get("custom_rules", [])
    cleaned = []
    report = {}
    
    for row in data:
        drop_reason = None
        
        for rule_name in rule_names:
            if rule_name not in registry:
                print("WARNING: unknown rule " + repr(rule_name))
                continue
            
            result = registry[rule_name](row, config)
            if result is not None:
                drop_reason = result
                break
        
        if drop_reason:
            report[drop_reason] = report.get(drop_reason, 0) + 1
        else:
            cleaned.append(row)
    
    print("Custom rules: " + str(len(data)) + " -> " + str(len(cleaned)) + " rows")
    for reason, count in report.items():
        print("  " + reason + ": " + str(count))
    
    return cleaned, report

# Test
data = [
    {"id": 1, "value": 25.0, "score": 80.0},
    {"id": 2, "value": -5.0, "score": 90.0},   # negative
    {"id": 3, "value": 50.0, "score": 70.0},
]

config = create_cleaning_config()
config["custom_rules"] = ["remove_negative"]
cleaned, rule_report = apply_custom_rules(data, config)

**Expected Output:**
```
Custom rules: 3 -> 2 rows
  negative_value: 1
```

### Try It Yourself

Write your own custom rule: `rule_check_timestamp` that rejects rows where the 'timestamp' field does not match the format 'YYYY-MM-DD'. Add it to RULE_REGISTRY and test it.

In [ ]:
# Try it: write a custom timestamp rule
# def rule_check_timestamp(row, config):
#     ...


### Common Mistakes: Cleaning Pipeline

**Mistake 1:** Modifying the original data -- `row[col] = val` changes the original dict.

**Fix:** Use `row = dict(row)` or `row = {**row}` to make a copy if you need the original.

**Mistake 2:** Not tracking drop reasons -- you drop 30% of rows but do not know why.

**Fix:** Always record the reason for every dropped row. This is your data quality report.

**Mistake 3:** Hard-coding rules instead of using config.

**Fix:** Put thresholds and column names in config. Your code reads from config.


### Key Takeaway

- Config-driven cleaning separates WHAT (config) from HOW (code)
- The rule registry maps string names to functions for easy extensibility
- Always track WHY each row was dropped
- Custom rules have a standard signature: (row, config) -> reason or None
- You can add new rules without changing the core cleaning code

---
## Mini-Quiz

In [ ]:
# Q1: Why is config-driven cleaning better than hard-coded rules?
# Answer: 

# Q2: What is the rule registry pattern?
# Answer: 

# Q3: What should a custom rule function return if the row is OK?
# Answer: 

---
## Homework: 12 Exercises

### Review (1-4)

In [ ]:
# HW1: Create a cleaning config for YOUR track project.


In [ ]:
# HW2: Write apply_cleaning_rules for your project data.
# Test with at least 10 rows including some that should be dropped.


In [ ]:
# HW3: Write 2 custom rules specific to your track.


In [ ]:
# HW4: Explain: why track drop reasons?


### Practice (5-8)

In [ ]:
# HW5: Write a rule that removes rows where a string column
# is longer than a configured max length.


In [ ]:
# HW6: Write a "cleaning pipeline" function that combines
# basic cleaning AND custom rules in sequence.


In [ ]:
# HW7: Add a "dry run" mode that reports what WOULD be dropped
# without actually removing rows.


In [ ]:
# HW8: Write a config loader that reads cleaning config from
# a JSON file.


### Challenge (9-11)

In [ ]:
# HW9: Implement rule priorities -- some rules should run before others.


In [ ]:
# HW10: Add a "fix instead of drop" option for certain rules.


In [ ]:
# HW11: Write a cleaning pipeline that produces a detailed log
# of every decision (kept/dropped/fixed) for every row.


### Mini-Project

In [ ]:
# HW12: Build a complete configurable cleaning module for your project.
# Requirements:
# - Config dict with at least 6 settings
# - 3+ custom rules in a registry
# - Cleaning report with counts and percentages
# - Demo with 20+ rows showing different drop reasons


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)